# TruthLens — Fine-tune a transformer classifier

Fine-tunes **DistilBERT** (or **MuRIL** for Indian languages) to classify news as real or fake.
Runs on any CUDA GPU (local RTX card or Colab: *Runtime → Change runtime type → GPU*). ~10 min on an RTX 4070.

Data: the same Kaggle *Fake and Real News* dataset as `train_classifier.ipynb` (`Fake.csv`, `True.csv`).

Output: a folder `fake_news_transformer/` → copy it to the project's `models/transformer/`.
Label convention: **1 = real, 0 = fake**.

> **MuRIL** (`google/muril-base-cased`) understands 17 Indian languages, but only helps if you train it on
> Indian-language data with real/fake labels (e.g. IFND). On this English-only dataset use DistilBERT.

In [ ]:
MODEL_NAME = "distilbert-base-uncased"   # or "google/muril-base-cased" with Indian-language data
MAX_LEN = 256        # tokens per article (the opening of an article carries most of the signal)
BATCH_SIZE = 32
EPOCHS = 2
LR = 3e-5
SEED = 42
OUT_DIR = "fake_news_transformer"


In [ ]:
# Colab only: install and upload the data
try:
    import google.colab  # noqa: F401
    !pip -q install transformers
    from google.colab import files
    files.upload()   # select Fake.csv and True.csv
except ImportError:
    print("Not on Colab — make sure Fake.csv and True.csv are in this folder.")

In [ ]:
import json, re, time
import numpy as np, pandas as pd, torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

torch.manual_seed(SEED); np.random.seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device, torch.cuda.get_device_name(0) if device == "cuda" else "(no GPU: this will be very slow)")

## 1. Load and clean the data
Same cleaning as the TF-IDF model: strip source quirks (Reuters datelines, tweets, "Getty Images", weekdays) so the model learns the writing, not the website.

In [ ]:
# KEEP IN SYNC with clean_text() in modules/text_classifier.py — the app applies the same cleaning.
LEAKS = re.compile(r'''
    https?://\S+ | www\.\S+ | pic\.twitter\.com/\S+ | \S+\.com\b | [@#]\w+
  | \breuters\b | \bgetty\s+images?\b | \bfeatured\s+image\b | \bimage\s+(via|credit)\b
  | \bphoto\s+(by|via|credit)\b | \bvia\s+(twitter|youtube|facebook)\b | \b21st\s+century\s+wire\b
  | \b(video|watch|screenshot|image|images)\b
  | \b(monday|tuesday|wednesday|thursday|friday|saturday|sunday)\b
''', re.I | re.X)

def clean_text(text):
    return re.sub(r"\s+", " ", LEAKS.sub(" ", str(text))).strip()

fake = pd.read_csv("Fake.csv"); real = pd.read_csv("True.csv")
fake["label"] = 0; real["label"] = 1
real["text"] = real["text"].str.replace(r"^.*?\(Reuters\)\s*-\s*", "", regex=True)
df = pd.concat([fake, real], ignore_index=True)
df["content"] = (df["title"].fillna("") + ". " + df["text"].fillna("")).map(clean_text)
df = df[df["content"].str.split().str.len() > 20].drop_duplicates("content").reset_index(drop=True)

train_df, rest = train_test_split(df, test_size=0.2, random_state=SEED, stratify=df["label"])
val_df, test_df = train_test_split(rest, test_size=0.5, random_state=SEED, stratify=rest["label"])
print(len(train_df), "train /", len(val_df), "val /", len(test_df), "test")

## 2. Tokenize

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def make_loader(frame, shuffle):
    enc = tokenizer(list(frame["content"]), truncation=True, max_length=MAX_LEN, padding="max_length", return_tensors="pt")
    ds = torch.utils.data.TensorDataset(enc["input_ids"], enc["attention_mask"], torch.tensor(frame["label"].values))
    return torch.utils.data.DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle)

train_loader, val_loader, test_loader = make_loader(train_df, True), make_loader(val_df, False), make_loader(test_df, False)

## 3. Fine-tune

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2, id2label={0: "fake", 1: "real"}, label2id={"fake": 0, "real": 1}
).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, int(0.06 * total_steps), total_steps)
use_amp = device == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

def evaluate(loader):
    model.eval(); preds, probs = [], []
    with torch.no_grad():
        for ids, mask, _ in loader:
            with torch.autocast(device_type=device, enabled=use_amp):
                logits = model(input_ids=ids.to(device), attention_mask=mask.to(device)).logits
            p = torch.softmax(logits.float(), -1)[:, 1]
            probs += p.tolist(); preds += (p >= 0.5).long().tolist()
    return np.array(preds), np.array(probs)

for epoch in range(EPOCHS):
    model.train(); t0 = time.time(); running = 0.0
    for step, (ids, mask, labels) in enumerate(train_loader, 1):
        optimizer.zero_grad()
        with torch.autocast(device_type=device, enabled=use_amp):
            loss = model(input_ids=ids.to(device), attention_mask=mask.to(device), labels=labels.to(device)).loss
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer); scaler.update(); scheduler.step()
        running += loss.item()
        if step % 200 == 0:
            print(f"  epoch {epoch+1} step {step}/{len(train_loader)}  loss {running/200:.4f}  ({time.time()-t0:.0f}s)")
            running = 0.0
    val_preds, _ = evaluate(val_loader)
    print(f"Epoch {epoch+1}: val accuracy {accuracy_score(val_df['label'], val_preds):.4f}  ({time.time()-t0:.0f}s)")

## 4. Test

In [ ]:
test_preds, test_probs = evaluate(test_loader)
test_acc = accuracy_score(test_df["label"], test_preds)
print(classification_report(test_df["label"], test_preds, target_names=["fake", "real"], digits=4))
print(confusion_matrix(test_df["label"], test_preds))

## 5. Save

In [ ]:
model.save_pretrained(OUT_DIR); tokenizer.save_pretrained(OUT_DIR)
json.dump({"base_model": MODEL_NAME, "max_len": MAX_LEN, "test_accuracy": round(test_acc, 4),
           "train_size": len(train_df), "dataset": "Kaggle Fake and Real News (ISOT)"},
          open(f"{OUT_DIR}/truthlens.json", "w"), indent=2)
print("Saved to", OUT_DIR)

try:  # Colab: download as a zip
    from google.colab import files
    import shutil; shutil.make_archive(OUT_DIR, "zip", OUT_DIR); files.download(OUT_DIR + ".zip")
except ImportError:
    print("Move this folder to models/transformer/ in the TruthLens project.")